# Assignment 17 — Bill Management Agent (Group Chat Collaboration with Bill Images, AG2)

**Objective:** Develop a Bill Management Agent that helps users track expenses by processing **bill images**, categorizing expenses into common groups, calculating total expenditure, and providing insights into spending patterns, using **Group Chat agent collaboration**. A **User Proxy Agent** initiates the conversation with the **Group Manager** and hands over the bill images. The **Bill Processing Agent** (vision) extracts and categorizes every expense with the total spent. The **Expense Summarization Agent** then compiles the total per category, names the highest-spending category, and highlights unusual or high spending.

**Framework:** AG2 `1.1.2` (`pip install ag2[openai]`), the current AG2 release. Images are passed to the model with AG2's `ImageInput`, which the OpenAI client sends as base64 image content. The group chat and the sequential routing are the same small plain-Python `GroupChat` / `GroupChatManager` / `state_transition` pattern the course teaches (AG2 Classic's `GroupChat`, `GroupChatManager` and `speaker_selection_method`), built on AG2 1.x agents:

| Assignment concept | AG2 Classic (course handouts) | This notebook (AG2 1.x) |
|---|---|---|
| User Proxy shares the image with the Group Manager | `user_proxy.initiate_chat(manager, message=[text + image])` | `UserProxyAgent.initiate_chat(manager, message, images)` |
| Multimodal agent | `MultimodalConversableAgent` | `Agent(...)` + `ImageInput(path=...)` passed to `ask()` |
| Group Chat / Group Manager | `GroupChat` + `GroupChatManager` | `GroupChat` (shared messages + images) + `GroupChatManager` (rounds, speaker selection) |
| Sequential flow | `speaker_selection_method=state_transition` | Same function: User Proxy → Bill Processing → Expense Summarization → end |

### How to run (Google Colab)
1. **Colab Secrets (recommended):** click the 🔑 icon in the left sidebar, add a secret named `OPENAI_API_KEY`, and turn on *Notebook access*.
2. **No secret?** Just run the notebook. If no key is found you will be prompted to paste one (hidden input, never printed or saved). An `OPENAI_API_KEY` environment variable also works.
3. `Runtime → Run all`. By default three **sample bill images** are generated so the notebook runs unattended. To use your own bills, set `USE_SAMPLE_BILLS = False` in the settings cell and upload them when prompted. The last cell prints a PASS/FAIL rubric.

### Rubric (verified automatically in the last cell)

| # | Expectation (from the assignment) | Where it is satisfied |
|---|---|---|
| R1 | **User Proxy Agent** obtains the bill images (validated: existing files, image extension, size limit; bad input rejected) | Bills cell + User Proxy self-test |
| R2 | User Proxy **initiates the conversation with the Group Manager** with a message about the images to be processed | Run cell (first group-chat message) |
| R3 | Group Manager calls the **Bill Processing Agent first and gives it the images** (image inputs recorded in the model request) | Manager log |
| R4 | Bill Processing Agent returns a **categorized list of all expenses** (allowed categories, amounts) | Bill Processing output (JSON block) |
| R5 | Bill Processing Agent's **total equals the sum of the extracted items** | Rubric (deterministic re-computation) |
| R6 | Extracted total matches the **sample bills' known amounts** (within 1%; skipped for your own bills) | Rubric (ground truth) |
| R7 | **Processed data moves to the Expense Summarization Agent** (it sees the categorized list) | Manager log |
| R8 | Summarization gives a **breakdown of total spending per category** (every category named) | Summarization output |
| R9 | Summarization names the **highest-expenditure category** correctly (matches deterministic computation) | Rubric |
| R10 | Summarization **highlights unusual or high spending trends** | Summarization output (Insights section) |
| R11 | **Sequential conversational experience** through the group chat: Bill Processing → Summarization, every speaker sees prior messages, ends cleanly | Manager log |
| R12 | **Final output displayed** (categorized expenses + spending summary in Markdown) | Final output cell |


## 1. Install dependencies
Installed with **uv** (fast). Versions are pinned so the notebook stays reproducible. Pillow is used only to draw the sample bills.

In [ ]:
import sys
%pip install -q uv
!uv pip install -q --python "{sys.executable}" "ag2[openai]==1.1.2" "pillow>=10.1"

import ag2
print("AG2 version:", ag2.__version__)

## 2. API key
Looks for the key in **Colab Secrets** first, then an environment variable, then asks you to paste it. The key is never printed.

In [ ]:
import os
from getpass import getpass


def load_api_key() -> str:
    try:
        from google.colab import userdata  # only exists inside Colab
        key = userdata.get("OPENAI_API_KEY")
        if key:
            return key
    except Exception:
        pass  # not in Colab, secret missing, or notebook access not granted
    key = os.environ.get("OPENAI_API_KEY")
    if key:
        return key
    return getpass("OPENAI_API_KEY not found in Colab Secrets or environment. Paste your key: ")


API_KEY = load_api_key().strip()
if not API_KEY:
    raise ValueError("No API key provided. Add OPENAI_API_KEY to Colab Secrets or paste it when prompted.")
print("API key loaded.")

## 3. Settings and model
All settings live here. The first reachable model in `MODEL_CANDIDATES` is used (all candidates accept images). Set `USE_SAMPLE_BILLS = False` to upload your own bill images instead of the generated samples.

In [ ]:
from ag2 import Agent
from ag2.config import OpenAIConfig

MODEL_CANDIDATES = ["gpt-5.4-mini", "gpt-5-mini", "gpt-4.1-mini"]  # vision-capable; tried in order, first reachable wins
MAX_ROUND = 4           # safety cap on group-chat rounds; the flow normally ends after 2
USE_SAMPLE_BILLS = True  # True: draw 3 sample bills. False: upload your own images (Colab) or put them in ./my_bills
MAX_IMAGE_MB = 10


async def pick_model() -> str:
    """Fail fast with a clear message if the key or every model name is wrong."""
    for model in MODEL_CANDIDATES:
        try:
            probe = Agent("probe", prompt="Reply with the single word: OK", config=OpenAIConfig(model=model, api_key=API_KEY))
            reply = await probe.ask("Ping")
            print(f"Model '{model}' reachable (test reply: {str(reply.body).strip()[:10]!r})")
            return model
        except Exception as exc:  # the provider error text can echo part of the key, so print only the error type
            print(f"Model '{model}' not usable ({type(exc).__name__}); trying the next candidate...")
    raise RuntimeError("No candidate model is reachable. Check that the API key is valid and MODEL_CANDIDATES "
                       "contains a model your account can use.")


MODEL = await pick_model()


def make_config() -> OpenAIConfig:
    """One config per agent. Change MODEL above to switch every agent at once."""
    return OpenAIConfig(model=MODEL, api_key=API_KEY)

## 4. Bill images
With `USE_SAMPLE_BILLS = True` three receipts (grocery, restaurant, utility bill) are drawn with Pillow, and their real totals are kept as **ground truth** so the rubric can check the extraction. Otherwise your own images are used.

In [ ]:
from glob import glob
from pathlib import Path

from IPython.display import Image as IPImage, display
from PIL import Image, ImageDraw, ImageFont

SAMPLE_BILLS = {  # merchant, line items; the totals below are the ground truth for rubric R6
    "bill_1_grocery.png": ("FreshMart Grocery", [("Milk 2L", 3.49), ("Eggs 12pk", 4.29), ("Bread", 2.99), ("Apples 1kg", 5.20), ("Chicken breast", 9.75)]),
    "bill_2_restaurant.png": ("Bella Pasta Restaurant", [("Pasta Carbonara", 14.50), ("Garden Salad", 8.00), ("Drinks x2", 6.00), ("Tip", 5.70)]),
    "bill_3_utility.png": ("City Power & Water", [("Electricity (Sept)", 62.40), ("Water (Sept)", 18.10), ("Service fee", 4.50)]),
}
SAMPLE_TRUTH = {name: round(sum(a for _, a in items), 2) for name, (_, items) in SAMPLE_BILLS.items()}


def render_bill(path: Path, merchant: str, items: list) -> None:
    """Draw a clean receipt-style image so a vision model can read it reliably."""
    try:
        font = ImageFont.load_default(size=26)
    except TypeError:  # very old Pillow
        font = ImageFont.load_default()
    lines = [merchant, "Date: 2026-10-15", "-" * 30] + [f"{name:<22}{amount:>8.2f}" for name, amount in items]
    lines += ["-" * 30, f"{'TOTAL':<22}{sum(a for _, a in items):>8.2f}", "Thank you!"]
    img = Image.new("RGB", (560, 60 + 40 * len(lines)), "white")
    draw = ImageDraw.Draw(img)
    for i, line in enumerate(lines):
        draw.text((30, 30 + 40 * i), line, fill="black", font=font)
    img.save(path)


if USE_SAMPLE_BILLS:
    bills_dir = Path("sample_bills")
    bills_dir.mkdir(exist_ok=True)
    for name, (merchant, items) in SAMPLE_BILLS.items():
        render_bill(bills_dir / name, merchant, items)
    BILL_IMAGE_PATHS = [str(bills_dir / name) for name in SAMPLE_BILLS]
    print("Sample bills generated. Ground-truth totals:", SAMPLE_TRUTH, "| grand total:", round(sum(SAMPLE_TRUTH.values()), 2))
else:
    try:
        from google.colab import files  # Colab: upload dialog
        BILL_IMAGE_PATHS = list(files.upload().keys())
    except ImportError:  # local: drop images into ./my_bills
        BILL_IMAGE_PATHS = sorted(glob("my_bills/*.*"))
    print("Using your bills:", BILL_IMAGE_PATHS)

for p in BILL_IMAGE_PATHS:
    display(IPImage(filename=p, width=320))

## 5. User Proxy Agent
Validates the bill images (no LLM: file exists, image extension, size limit) and **initiates the conversation with the Group Manager**, passing the images along.

In [ ]:
from ag2.events import BinaryInput, ImageInput

ALLOWED_EXT = (".png", ".jpg", ".jpeg", ".webp", ".gif")


def validate_bill_images(paths: list) -> list:
    """Deterministic validation of user-supplied files before anything is sent to the model."""
    if not paths:
        raise ValueError("No bill images provided")
    valid = []
    for p in paths:
        path = Path(p)
        if not path.is_file():
            raise ValueError(f"Bill image not found: {p}")
        if path.suffix.lower() not in ALLOWED_EXT:
            raise ValueError(f"Unsupported file type {path.suffix!r} for {p}; allowed: {ALLOWED_EXT}")
        if path.stat().st_size > MAX_IMAGE_MB * 1024 * 1024:
            raise ValueError(f"{p} is larger than {MAX_IMAGE_MB} MB")
        valid.append(path)
    return valid


class UserProxyAgent:
    """Holds the user's bill images (no LLM) and initiates the group chat."""

    name = "User_Proxy_Agent"

    def __init__(self, image_paths: list):
        self.image_paths = validate_bill_images(image_paths)

    async def initiate_chat(self, manager, message: str):
        images = [ImageInput(path=p) for p in self.image_paths]
        return await manager.run(sender=self, message=message, images=images)


# Self-test: the sample images pass; a missing file, a wrong extension and an empty list must fail.
Path("not_an_image.txt").write_text("hello")
VALIDATION_SHOULD_FAIL_OK = True
for bad in [["missing_bill.png"], ["not_an_image.txt"], []]:
    try:
        validate_bill_images(bad)
        VALIDATION_SHOULD_FAIL_OK = False
        print("FAIL: accepted bad input", bad)
    except ValueError:
        pass
assert len(validate_bill_images(BILL_IMAGE_PATHS)) == len(BILL_IMAGE_PATHS)
print("User Proxy Agent defined; validation self-test:", "PASS" if VALIDATION_SHOULD_FAIL_OK else "FAIL")

## 6. Bill Processing Agent and Expense Summarization Agent
The Bill Processing Agent reads the images and must return both a readable table and a machine-readable JSON block (fixed categories) so the totals can be verified deterministically. The Summarization Agent works only from that categorized list.

In [ ]:
BILL_AGENT = "Bill_Processing_Agent"
SUMMARY_AGENT = "Expense_Summarization_Agent"
CATEGORIES = ("Groceries", "Dining", "Utilities", "Shopping", "Entertainment", "Transport", "Healthcare", "Other")

SHARED_RULES = ("You are taking part in a group chat managed by a Group Manager; you see the full transcript. "
                "Reply only with your own contribution in Markdown. Do not invent data that is not in the bills.")

bill_processing_agent = Agent(
    BILL_AGENT,
    prompt=f"""You are the Bill Processing Agent. {SHARED_RULES}
Task: the user's bill/receipt images are attached. For EVERY image, read the merchant and every line item with its amount
(ignore subtotals, tax lines that are already included in the total, and tips only if listed as a separate line item).
Assign each item exactly one category from: {", ".join(CATEGORIES)}.
Reply with:
1. A Markdown table with columns Bill | Merchant | Item | Category | Amount.
2. The line '**Total amount spent:** <sum of all items, 2 decimals>'.
3. A fenced ```json block with this exact shape (amounts as numbers, categories from the list above):
{{"expenses": [{{"bill": "<image file name or Bill 1/2/3>", "merchant": "...", "item": "...", "category": "...", "amount": 0.0}}], "total": 0.0}}""",
    config=make_config(),
)

expense_summarization_agent = Agent(
    SUMMARY_AGENT,
    prompt=f"""You are the Expense Summarization Agent. {SHARED_RULES}
Task: analyze the categorized expense list produced by the Bill Processing Agent (use its JSON block as the source of truth).
Reply with exactly these sections:
### Spending by category
A Markdown table: Category | Total | Share of spending (%), one row per category present, sorted by total descending,
plus a final row for the grand total.
**Highest category:** <the category with the largest total>
### Insights
3-5 bullet points on spending patterns: highlight unusual or high spending, the largest single item, and one practical
saving suggestion. Keep the whole reply under 250 words.""",
    config=make_config(),
)
print("Agents ready:", bill_processing_agent.name, "|", expense_summarization_agent.name)

## 7. Group Chat and Group Manager
`GroupChat` holds the shared messages and the user's images. `GroupChatManager` runs the rounds: it asks the state-transition function who speaks next, sends that agent the full transcript (plus the images for the vision agent), appends the reply, and stops when the function returns `None` or `max_round` is reached. The number of image inputs actually sent is read back from AG2's own `ModelRequest` record.

In [ ]:
from ag2.events import ModelRequest


def state_transition(last_speaker, groupchat):
    """Sequential flow: User Proxy -> Bill Processing -> Expense Summarization -> end."""
    if isinstance(last_speaker, UserProxyAgent):
        return bill_processing_agent
    if last_speaker is bill_processing_agent:
        return expense_summarization_agent
    if last_speaker is expense_summarization_agent:
        return None
    raise ValueError(f"No transition for speaker {getattr(last_speaker, 'name', last_speaker)!r}")


class GroupChat:
    """Shared conversation: every speaker receives the full transcript."""

    def __init__(self, agents: list, max_round: int, speaker_selection_method, vision_agents: set):
        self.agents = agents
        self.max_round = max_round
        self.speaker_selection_method = speaker_selection_method
        self.vision_agents = vision_agents  # agents that receive the images
        self.messages: list[dict] = []
        self.images: list = []

    def transcript(self) -> str:
        return "\n\n".join(f"[{m['name']}]:\n{m['content']}" for m in self.messages)


class GroupChatManager:
    """Manages the group chat and directs the conversation flow."""

    name = "Group_Manager"

    def __init__(self, groupchat: GroupChat):
        self.groupchat = groupchat
        self.speaker_order: list[str] = []
        self.ended_by_transition = False

    async def run(self, sender, message: str, images: list = ()) -> list[dict]:
        gc = self.groupchat
        gc.messages.clear(); gc.images = list(images); self.speaker_order.clear(); self.ended_by_transition = False
        line = "-" * 80
        gc.messages.append({"name": sender.name, "content": message, "visible_messages": 0, "images_sent": 0})
        print(f"{line}\n{sender.name} -> {self.name}  (+{len(gc.images)} image(s))\n{line}\n{message}\n")

        last_speaker = sender
        for round_no in range(1, gc.max_round + 1):
            speaker = gc.speaker_selection_method(last_speaker, gc)
            if speaker is None:
                self.ended_by_transition = True
                print(f"[{self.name}] Flow complete after {round_no - 1} rounds.")
                break
            visible = len(gc.messages)
            attach = gc.images if speaker.name in gc.vision_agents else []
            print(f"[{self.name}] Round {round_no}: next speaker -> {speaker.name} (sees {visible} messages, {len(attach)} image(s))")
            prompt = f"Group chat transcript so far:\n\n{gc.transcript()}\n\nIt is now your turn, {speaker.name}."
            reply = await speaker.ask(prompt, *attach)
            images_sent = sum(isinstance(p, BinaryInput) for e in await reply.history.get_events()
                              if isinstance(e, ModelRequest) for p in e.parts)
            gc.messages.append({"name": speaker.name, "content": reply.body or "", "visible_messages": visible, "images_sent": images_sent})
            self.speaker_order.append(speaker.name)
            print(f"{line}\n{speaker.name}\n{line}\n{reply.body}\n")
            last_speaker = speaker
        else:
            print(f"[{self.name}] max_round={gc.max_round} reached; stopping.")
        return gc.messages


group_chat = GroupChat(agents=[bill_processing_agent, expense_summarization_agent], max_round=MAX_ROUND,
                       speaker_selection_method=state_transition, vision_agents={BILL_AGENT})
group_manager = GroupChatManager(groupchat=group_chat)
print("Group chat ready with agents:", [a.name for a in group_chat.agents])

## 8. Run the conversation
The User Proxy Agent shares the bill images with the Group Manager; the manager drives the Bill Processing Agent and then the Expense Summarization Agent.

In [ ]:
user_proxy = UserProxyAgent(BILL_IMAGE_PATHS)

initial_message = (
    f"Please process my {len(user_proxy.image_paths)} bill image(s) attached to this message "
    f"({', '.join(p.name for p in user_proxy.image_paths)}): extract every expense, categorize them, "
    "calculate the total expenditure and give me insights into my spending."
)

messages = await user_proxy.initiate_chat(group_manager, message=initial_message)
print("Speaker order:", " -> ".join(group_manager.speaker_order))

## 9. Final output — categorized expenses and spending summary

In [ ]:
import json
import re

from IPython.display import Markdown

bill_output = next((m["content"] for m in messages if m["name"] == BILL_AGENT), "")
summary_output = next((m["content"] for m in messages if m["name"] == SUMMARY_AGENT), "")


def parse_expenses(text: str) -> dict:
    """Pull the JSON block out of the Bill Processing Agent's reply (exact parsing, no LLM)."""
    match = re.search(r"```json\s*(\{.*?\})\s*```", text, flags=re.DOTALL)
    try:
        return json.loads(match.group(1)) if match else {}
    except json.JSONDecodeError:
        return {}


expenses = parse_expenses(bill_output)
by_category: dict = {}
for item in expenses.get("expenses", []):
    by_category[item.get("category", "Other")] = round(by_category.get(item.get("category", "Other"), 0) + float(item.get("amount", 0)), 2)
computed_total = round(sum(by_category.values()), 2)
highest_category = max(by_category, key=by_category.get) if by_category else None

display(Markdown(f"# Bill Management Report\n\n## 1. Categorized expenses ({BILL_AGENT})\n\n{bill_output}\n\n---\n\n"
                 f"## 2. Spending summary ({SUMMARY_AGENT})\n\n{summary_output}\n\n---\n\n"
                 f"*Deterministic check: {len(expenses.get('expenses', []))} items parsed, computed total {computed_total:.2f}, "
                 f"per category {by_category}, highest: {highest_category}*"))

## 10. Rubric verification
Automated check of the rubric table at the top. Every row should print `PASS`.

In [ ]:
order = group_manager.speaker_order
bill_msg = next((m for m in messages if m["name"] == BILL_AGENT), {})
summary_msg = next((m for m in messages if m["name"] == SUMMARY_AGENT), {})
items = expenses.get("expenses", [])
stated_total = float(expenses.get("total", -1)) if isinstance(expenses, dict) and "total" in expenses else -1.0
truth_total = round(sum(SAMPLE_TRUTH.values()), 2)
stated_highest = re.search(r"\*\*Highest category:\*\*\s*([A-Za-z]+)", summary_output or "")
text = lambda s: (s or "").lower()

checks = [
    ("R1  User Proxy holds validated bill images; bad input rejected (should-fail cases)",
     len(user_proxy.image_paths) >= 1 and VALIDATION_SHOULD_FAIL_OK),
    ("R2  User Proxy initiated the conversation with the Group Manager about the images",
     messages and messages[0]["name"] == user_proxy.name and "bill image" in text(messages[0]["content"])),
    ("R3  Group Manager called the Bill Processing Agent first and gave it the images",
     order[:1] == [BILL_AGENT] and bill_msg.get("images_sent") == len(user_proxy.image_paths)),
    ("R4  Bill Processing Agent returned a categorized expense list with allowed categories and amounts",
     len(items) >= 3 and all(i.get("category") in CATEGORIES and float(i.get("amount", 0)) > 0 for i in items)),
    ("R5  Stated total equals the sum of the extracted items", items and abs(stated_total - computed_total) < 0.01),
    ("R6  Extracted total matches the sample bills' ground truth (within 1%)" + ("" if USE_SAMPLE_BILLS else " [skipped: own bills]"),
     (not USE_SAMPLE_BILLS) or (items and abs(computed_total - truth_total) <= 0.01 * truth_total)),
    ("R7  Processed data moved to the Expense Summarization Agent (it saw the categorized list)",
     order[1:2] == [SUMMARY_AGENT] and summary_msg.get("visible_messages") == 2),
    ("R8  Summarization gives a per-category breakdown (every category named)",
     by_category and all(c.lower() in text(summary_output) for c in by_category) and "spending by category" in text(summary_output)),
    ("R9  Summarization's highest category matches the deterministic computation",
     bool(stated_highest) and highest_category and stated_highest.group(1).lower() == highest_category.lower()),
    ("R10 Summarization highlights spending trends / insights",
     "insights" in text(summary_output) and any(w in text(summary_output) for w in ("unusual", "high", "largest"))),
    ("R11 Sequential group-chat flow: Bill Processing -> Summarization, all prior messages visible, ended cleanly",
     order == [BILL_AGENT, SUMMARY_AGENT] and group_manager.ended_by_transition
     and all(m["visible_messages"] == i for i, m in enumerate(messages))),
    ("R12 Final output (categorized expenses + summary) displayed as Markdown", bool(bill_output.strip()) and bool(summary_output.strip())),
]

for name, ok in checks:
    print(f"{'PASS' if ok else 'FAIL'}  {name}")
print(f"\n{sum(bool(ok) for _, ok in checks)}/{len(checks)} checks passed")